In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for use! \n\nI’m designed to take text and images as input and generate text as output. \n\nHow can I help you today?'}]}]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for anyone to use! 

I can take text and images as input and provide text-based responses. 

What’s up? 😊<end_of_turn>


In [12]:
%%writefile scorer.py

import re
import sys
import unicodedata

def normalize(s: str) -> str:
    """Lowercase, strip unicode quirks, remove trailing punctuation, collapse whitespace."""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = s.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    s = re.sub(r"^(answer|the answer is)\s*[:\-]?\s*", "", s)
    s = re.sub(r"[.,;:!?]+$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_number(s: str):
    """Canonicalize a numeric string to 3-decimal form. Returns None if not numeric.
    Accepts '0.312', '.312', '0.3120', '31.2%' -> all become '0.312'.
    """
    s = s.strip()
    pct = s.endswith("%")
    if pct:
        s = s[:-1].strip()
    m = re.fullmatch(r"(-?\d*\.?\d+)", s)
    if not m:
        return None
    try:
        val = float(m.group(1))
    except ValueError:
        return None
    if pct:
        val /= 100.0
    return f" {val:.3f}".strip()

def score_mc(pred: str, gold: str) -> bool:
    """Gold is a single letter A/B/C/D. Take the first standalone A/B/C/D in the prediction."""
    m = re.search(r"\b([ABCD])\b", pred.upper())
    return bool(m) and m.group(1) == gold.strip().upper()

def score_short(pred: str, gold: str) -> bool:
    """Exact match after normalization; numeric values compared to 3 decimals."""
    p, g = normalize(pred), normalize(gold)
    if p == g:
        return True
    pn, gn = normalize_number(p), normalize_number(g)
    if pn is not None and gn is not None:
        return pn == gn
    return False

def score(pred: str, gold: str, qtype: str) -> bool:
    if qtype == "mc":
        return score_mc(pred, gold)
    # "short" and "stats" both go through score_short
    return score_short(pred, gold)

def _run_tests():
    cases_short = [
        ("0.312", "0.312", True),
        (".312", "0.312", True),
        ("31.2%", "0.312", True),
        ("0.3120", "0.312", True),
        ("The answer is 0.312.", "0.312", True),
        ("0.31", "0.312", False),
        ("Babe Ruth", "babe ruth", True),
        ("Ruth", "Babe Ruth", False),
        ("  Babe   Ruth  ", "Babe Ruth", True),
    ]
    cases_mc = [
        ("A", "A", True),
        ("a.", "A", True),
        ("The answer is C", "C", True),
        ("B because ...", "B", True),
        ("D", "A", False),
        ("I don't know", "A", False),
    ]

    failures = 0
    for pred, gold, expected in cases_short:
        got = score_short(pred, gold)
        if got != expected:
            failures += 1
            print(f"FAIL short: pred={pred!r} gold={gold!r} expected={expected} got={got}")
    for pred, gold, expected in cases_mc:
        got = score_mc(pred, gold)
        if got != expected:
            failures += 1
            print(f"FAIL mc:    pred={pred!r} gold={gold!r} expected={expected} got={got}")

    if failures == 0:
        print(f"All {len(cases_short) + len(cases_mc)} scorer tests passed.")
    else:
        print(f"{failures} test(s) FAILED.")
    return failures == 0

def test_short():
    for pred, gold, expected in [
        ("0.312", "0.312", True), (".312", "0.312", True),
        ("31.2%", "0.312", True), ("0.3120", "0.312", True),
        ("The answer is 0.312.", "0.312", True), ("0.31", "0.312", False),
        ("Babe Ruth", "babe ruth", True), ("Ruth", "Babe Ruth", False),
        ("  Babe   Ruth  ", "Babe Ruth", True),
    ]:
        assert score_short(pred, gold) == expected, (pred, gold, expected)


def test_mc():
    for pred, gold, expected in [
        ("A", "A", True), ("a.", "A", True),
        ("The answer is C", "C", True), ("B because ...", "B", True),
        ("D", "A", False), ("I don't know", "A", False),
    ]:
        assert score_mc(pred, gold) == expected, (pred, gold, expected)


if __name__ == "__main__":
    ok = _run_tests()
    if 'ipykernel' not in sys.modules:
        sys.exit(0 if ok else 1)

Writing scorer.py


In [ ]:

import os, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

from scorer import score

MODEL_ID = "google/gemma-3-1b-it"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

KNOWLEDGE_ITEMS = [
    # (question, gold, qtype)
    ("How many players are on the field for one team in baseball at the start of a play?\nA. 8\nB. 9\nC. 10\nD. 11", "B", "mc"),
    ("How many strikes make a strikeout?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
    ("How many balls make a walk?\nA. 3\nB. 4\nC. 5\nD. 6", "B", "mc"),
    ("Which team won the 2016 World Series?\nA. Cleveland Indians\nB. Chicago Cubs\nC. Boston Red Sox\nD. Los Angeles Dodgers", "B", "mc"),
    ("Who holds the MLB career home run record?", "Barry Bonds", "short"),
    ("How many innings are in a standard MLB game?\nA. 7\nB. 8\nC. 9\nD. 10", "C", "mc"),
    ("What is it called when a batter hits and reaches all four bases in one play?\nA. Triple\nB. Home run\nC. Grand slam\nD. Cycle", "B", "mc"),
    ("Which league uses the designated hitter in all games as of 2022?\nA. National League\nB. American League\nC. Both\nD. Neither", "C", "mc"),
    ("Who is the all-time MLB hits leader?", "Pete Rose", "short"),
    ("How many outs end a half-inning?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
]

# 10 stats questions. Counts are embedded -> pure calculation, per the plan.
STATS_ITEMS = [
    ("Using these career totals: AB=1000, H=300, BB=100, HBP=5, SF=5. What is the career OBP? Round to 3 decimals.", "0.370", "stats"),
    ("Using these career totals: AB=800, H=200, BB=80, HBP=0, SF=0. What is the career OBP? Round to 3 decimals.", "0.318", "stats"),
    ("Using these career totals: AB=500, H=150, 2B=30, 3B=5, HR=20. What is the career SLG? Round to 3 decimals.", "0.500", "stats"),
    ("Using these career totals: AB=400, H=100, 2B=20, 3B=2, HR=10. What is the career SLG? Round to 3 decimals.", "0.385", "stats"),
    ("Using these career totals: AB=600, H=180. What is the career AVG? Round to 3 decimals.", "0.300", "stats"),
    ("Using these career totals: AB=900, H=270, BB=90, HBP=5, SF=5. What is the career OBP? Round to 3 decimals.", "0.370", "stats"),
    ("Using these career totals: AB=700, H=210, 2B=40, 3B=4, HR=30. What is the career SLG? Round to 3 decimals.", "0.497", "stats"),
    ("Using these career totals: AB=300, H=90. What is the career AVG? Round to 3 decimals.", "0.300", "stats"),
    ("Using these career totals: AB=1000, H=250, BB=150, HBP=10, SF=10. What is the career OBP? Round to 3 decimals.", "0.350", "stats"),
    ("Using these career totals: AB=550, H=140, 2B=25, 3B=3, HR=15. What is the career SLG? Round to 3 decimals.", "0.393", "stats"),
]

print("Loading tokenizer + model (fp16 on T4)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=userdata.get("HF_TOKEN"),
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Loaded. GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def generate(prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run(items, label):
    correct, results = 0, []
    t0 = time.time()
    for q, gold, qtype in items:
        pred = generate(q)
        ok = score(pred, gold, qtype)
        correct += ok
        results.append((q[:60], gold, pred[:60], ok))
    dt = time.time() - t0
    acc = correct / len(items)
    print(f"\n=== {label}: {correct}/{len(items)} = {acc:.0%}  ({dt:.1f}s) ===")
    for q, gold, pred, ok in results:
        mark = "✓" if ok else "✗"
        print(f"  {mark} gold={gold!r:>14}  pred={pred!r}")
    return acc

print("\n" + "="*60)
acc_know = run(KNOWLEDGE_ITEMS, "KNOWLEDGE (base, zero-shot)")
acc_stats = run(STATS_ITEMS,    "STATS (base, zero-shot, counts in prompt)")

print("\n" + "="*60)
print(f"Knowledge accuracy: {acc_know:.0%}")
print(f"Stats accuracy:     {acc_stats:.0%}")

if acc_know < 0.20:
    print("\nVERDICT: knowledge is near the floor (<20%).")
    print("  -> A 1B model likely lacks the latent knowledge to 'unlock'.")
    print("  -> SWITCH to a structured classification/extraction task type.")
elif acc_know > 0.85:
    print("\nVERDICT: knowledge is too easy (>85%).")
    print("  -> Base already strong; LoRA delta will be hard to see.")
    print("  -> ADD harder items or change task type.")
else:
    print("\nVERDICT: knowledge is in the 20-85% band — a usable 'middle'.")
    print("  -> KEEP baseball; cut stats to ~10-15% of test, report separately.")
    print("  -> Proceed to dataset build.")
print(f"\nGPU peak so far: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Loading tokenizer + model (fp16 on T4)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded. GPU mem after load: 6.06 GB


=== KNOWLEDGE (base, zero-shot): 6/10 = 60%  (25.6s) ===
  ✓ gold=           'B'  pred="The correct answer is **B. 9**\n\nHere's why:\n\n*   **Starting "
  ✓ gold=           'B'  pred='The correct answer is **B. 3**\n\nA strikeout is when a pitche'
  ✓ gold=           'B'  pred='The correct answer is **B. 4**\n\nA walk is typically made up '
  ✓ gold=           'B'  pred='The correct answer is **B. Chicago Cubs**.\n\nThe Chicago Cubs'
  ✗ gold= 'Barry Bonds'  pred='As of today, November 2, 2023, the MLB career home run recor'
  ✗ gold=           'C'  pred="The correct answer is **B. 8**\n\nHere's how innings are calcu"
  ✗ gold=           'B'  pred="The correct answer is **D. Cycle**.\n\nHere's why:\n\n*   **Cycl"
  ✓ gold=           'C'  pred='The correct answer is **C. Both**\n\nAs of 2022, the MLB utili'
  ✗ gold=   'Pete Rose'  pred='As of today, November 2, 2023, the all-time MLB hits leader '
  ✓ gold=           'B'  pred="The correct answer 

In [ ]:
# probe_base_model_v2.py — revised 10-item probe
# bare-answer prompting for short items; formulas defined

import os, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scorer import score

MODEL_ID = "google/gemma-3-1b-it"
HF_TOKEN = os.environ.get("HF_TOKEN") or "PASTE_YOUR_TOKEN_HERE"

BARE = " Answer with just the name or phrase, no explanation."
NUM  = " Give only the number, rounded to 3 decimals, no explanation."

KNOWLEDGE_ITEMS = [
    ("How many players are on the field for one team in baseball at the start of a play?\nA. 8\nB. 9\nC. 10\nD. 11", "B", "mc"),
    ("How many strikes make a strikeout?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
    ("How many balls make a walk?\nA. 3\nB. 4\nC. 5\nD. 6", "B", "mc"),
    ("Which team won the 2016 World Series?\nA. Cleveland Indians\nB. Chicago Cubs\nC. Boston Red Sox\nD. Los Angeles Dodgers", "B", "mc"),
    ("Who holds the MLB career home run record?" + BARE, "Barry Bonds", "short"),
    ("How many innings are in a standard MLB game?\nA. 7\nB. 8\nC. 9\nD. 10", "C", "mc"),
    ("What is it called when a batter hits and reaches all four bases in one play?\nA. Triple\nB. Home run\nC. Grand slam\nD. Cycle", "B", "mc"),
    ("Which league uses the designated hitter in all games as of 2022?\nA. National League\nB. American League\nC. Both\nD. Neither", "C", "mc"),
    ("Who is the all-time MLB hits leader?" + BARE, "Pete Rose", "short"),
    ("How many outs end a half-inning?\nA. 2\nB. 3\nC. 4\nD. 5", "B", "mc"),
]

OBP_DEF = "OBP = (H + BB + HBP) / (AB + BB + HBP + SF)."
SLG_DEF = "SLG = (singles + 2*2B + 3*3B + 4*HR) / AB, where singles = H - 2B - 3B - HR."
AVG_DEF = "AVG = H / AB."

STATS_ITEMS = [
    (f"Using {OBP_DEF} Career totals: AB=1000, H=300, BB=100, HBP=5, SF=5. What is OBP?{NUM}", "0.365", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=800, H=200, BB=80, HBP=0, SF=0. What is OBP?{NUM}", "0.318", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=500, H=150, 2B=30, 3B=5, HR=20. What is SLG?{NUM}", "0.500", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=400, H=100, 2B=20, 3B=2, HR=10. What is SLG?{NUM}", "0.385", "stats"),
    (f"Using {AVG_DEF} Career totals: AB=600, H=180. What is AVG?{NUM}", "0.300", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=900, H=270, BB=90, HBP=5, SF=5. What is OBP?{NUM}", "0.367", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=700, H=210, 2B=40, 3B=4, HR=30. What is SLG?{NUM}", "0.497", "stats"),
    (f"Using {AVG_DEF} Career totals: AB=300, H=90. What is AVG?{NUM}", "0.300", "stats"),
    (f"Using {OBP_DEF} Career totals: AB=1000, H=250, BB=150, HBP=10, SF=10. What is OBP?{NUM}", "0.350", "stats"),
    (f"Using {SLG_DEF} Career totals: AB=550, H=140, 2B=25, 3B=3, HR=15. What is SLG?{NUM}", "0.393", "stats"),
]

def score_short_loose(pred: str, gold: str) -> bool:
    """Diagnostic only: gold appears as a substring after normalization."""
    from scorer import normalize
    return normalize(gold) in normalize(pred)

print("Loading tokenizer + model (fp16 on T4)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map="auto",
)
model.eval()
print(f"Loaded. GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def generate(prompt, max_new_tokens=96):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run(items, label):
    correct_strict, correct_loose, results = 0, 0, []
    t0 = time.time()
    for q, gold, qtype in items:
        pred = generate(q)
        ok_strict = score(pred, gold, qtype)
        ok_loose  = ok_strict or (qtype != "mc" and score_short_loose(pred, gold))
        correct_strict += ok_strict
        correct_loose  += ok_loose
        results.append((gold, pred, ok_strict, ok_loose, qtype))
    dt = time.time() - t0
    n = len(items)
    print(f"\n=== {label}: strict {correct_strict}/{n} = {correct_strict/n:.0%}"
          f" | loose {correct_loose}/{n} = {correct_loose/n:.0%}  ({dt:.1f}s) ===")
    for gold, pred, ok_s, ok_l, qtype in results:
        mark = "✓" if ok_s else ("~" if ok_l else "✗")   # ✓ strict, ~ loose-only, ✗ miss
        print(f"  {mark} [{qtype:5}] gold={gold!r:>14}  pred={pred[:70]!r}")
    return correct_strict / n, correct_loose / n

print("\n" + "="*60)
know_strict, know_loose = run(KNOWLEDGE_ITEMS, "KNOWLEDGE")
stats_strict, stats_loose = run(STATS_ITEMS, "STATS")

print("\n" + "="*60)
print(f"Knowledge: strict {know_strict:.0%} | loose {know_loose:.0%}")
print(f"Stats:     strict {stats_strict:.0%} | loose {stats_loose:.0%}")

print("\nInterpretation:")
if know_loose - know_strict >= 0.10:
    print(f"  Short items lift {know_loose-know_strict:.0%} with loose matching ->")
    print("  formatting was the miss. Use bare-answer prompts + strict scoring in the real set.")
else:
    print("  Small strict-vs-loose gap -> the short misses are knowledge, not format.")
    print("  Consider making the knowledge slice MC-dominant.")

if stats_strict >= 0.10:
    print(f"  Stats off the floor ({stats_strict:.0%}) with defined formulas ->")
    print("  keep a small stats control (~10-15% of test).")
else:
    print(f"  Stats still at {stats_strict:.0%} even with formulas defined ->")
    print("  the 1B model can't compute these. Keep 5-10 stats items purely as the")
    print("  caveat-demonstrator; do not make stats a headline metric.")

print(f"\nGPU peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Loading tokenizer + model (fp16 on T4)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded. GPU mem after load: 6.06 GB


=== KNOWLEDGE: strict 6/10 = 60% | loose 6/10 = 60%  (24.0s) ===
  ✓ [mc   ] gold=           'B'  pred="The correct answer is **B. 9**\n\nHere's why:\n\n*   **Starting Field Posi"
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. 3**\n\nA strikeout is when a pitcher throws a'
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. 4**\n\nA walk is typically made up of four ba'
  ✓ [mc   ] gold=           'B'  pred='The correct answer is **B. Chicago Cubs**.\n\nThe Chicago Cubs won the 2'
  ✗ [short] gold= 'Barry Bonds'  pred='Babe Ruth'
  ✗ [mc   ] gold=           'C'  pred="The correct answer is **B. 8**\n\nHere's how innings are calculated in a"
  ✗ [mc   ] gold=           'B'  pred="The correct answer is **D. Cycle**.\n\nHere's why:\n\n*   **Cycle:** This "
  ✓ [mc   ] gold=           'C'  pred='The correct answer is **C. Both**\n\nAs of 2022, the MLB utilized the de'
  ✗ [short] gold=   'Pete Rose'  pred='Babe Ruth'
  

In [7]:
# build_dataset.py — generate + freeze the baseball dataset
# Mix (of 430 total): 55% MC, 30% short, 15% stats
#   train 300 / val 30 / test 100
# Stats golds computed in code, never hand-written.
# Emits train.json / val.json / test.json + manifest.json
import json, random, hashlib, re
from pathlib import Path
from dataclasses import dataclass, asdict

SEED = 0
N_TRAIN, N_VAL, N_TEST = 24, 3, 9
MIX = {"mc": 0.65, "short": 0.35}

# TODO 1: point this at your Lahman data. Options:
#   - pybaseball: from pybaseball import batting; df = batting
#   - Lahman CSVs: read Batting.csv and People.csv
# Expected batting columns: playerID, AB, H, 2B, 3B, HR, BB, HBP, SF
# Expected people columns:  playerID, nameFirst, nameLast
BATTING_CSV = "/content/sample_data/Batting.csv"
PEOPLE_CSV  = "/content/sample_data/People.csv"

# TODO 2: hand-written MC + short items live in this file or a sibling.
# Each entry: {"qtype": "mc"|"short", "knowledge": true,
#              "question": str, "choices": [...] (mc only),
#              "answer_idx": int (mc only), "answer": str (short only)}
# You need roughly:
#   MC    ~165 train + ~17 val + ~55 test  -> ~237 unique MC items
#   short ~90 train  + ~9 val  + ~30 test  -> ~129 unique short items
# Write them; the generator handles splitting and formatting.
HANDWRITTEN_PATH = "/content/sample_data/handwritten_items.json"

def load_batting():
    import csv
    with open(BATTING_CSV) as f:
        rows = list(csv.DictReader(f))
    people = {}
    if PEOPLE_CSV:
        with open(PEOPLE_CSV) as f:
            for p in csv.DictReader(f):
                people[p["playerID"]] = f"{p['nameFirst']} {p['nameLast']}".strip()
    out = []
    for r in rows:
        try:
            rec = {k: int(r[k]) for k in ["AB","H","2B","3B","HR","BB"]}
        except (KeyError, ValueError):
            continue
        for k in ["HBP","SF"]:
            rec[k] = int(r[k]) if r.get(k) not in ("", None) else 0
        if rec["AB"] < 100:
            continue
        rec["name"] = people.get(r["playerID"], r["playerID"])
        out.append(rec)
    return out

OBP_DEF = "OBP = (H + BB + HBP) / (AB + BB + HBP + SF)."
SLG_DEF = "SLG = (singles + 2*2B + 3*3B + 4*HR) / AB, where singles = H - 2B - 3B - HR."
AVG_DEF = "AVG = H / AB."
NUM = " Give only the number, rounded to 3 decimals, no explanation."

def make_stats_items(players, n, rng, prefix="stats"):
    """Generate n stats items, golds computed from the same numbers in the prompt."""
    rng.shuffle(players)
    items, i = [], 0
    for p in players:
        if i >= n: break
        kind = ["obp","slg","avg"][i % 3]
        if kind == "obp":
            denom = p["AB"] + p["BB"] + p["HBP"] + p["SF"]
            if denom == 0: continue
            gold = f"{(p['H']+p['BB']+p['HBP'])/denom:.3f}"
            q = (f"Using {OBP_DEF} Career totals: AB={p['AB']}, H={p['H']}, "
                 f"BB={p['BB']}, HBP={p['HBP']}, SF={p['SF']}. What is OBP?{NUM}")
        elif kind == "slg":
            if p["AB"] == 0: continue
            singles = p["H"] - p["2B"] - p["3B"] - p["HR"]
            tb = singles + 2*p["2B"] + 3*p["3B"] + 4*p["HR"]
            gold = f"{tb/p['AB']:.3f}"
            q = (f"Using {SLG_DEF} Career totals: AB={p['AB']}, H={p['H']}, "
                 f"2B={p['2B']}, 3B={p['3B']}, HR={p['HR']}. What is SLG?{NUM}")
        else:
            if p["AB"] == 0: continue
            gold = f"{p['H']/p['AB']:.3f}"
            q = (f"Using {AVG_DEF} Career totals: AB={p['AB']}, H={p['H']}. What is AVG?{NUM}")
        items.append({"id": f"{prefix}_{i:03d}", "qtype": "stats", "knowledge": False,
                      "question": q, "answer": gold})
        i += 1
    return items

BARE = " Answer with just the name or phrase, no explanation."

def to_example(item):
    """Convert a raw item dict to the chat-format Example + gold + metadata."""
    if item["qtype"] == "mc":
        labels = "ABCD"
        prompt = item["question"] + "\n" + "\n".join(
            f"{labels[j]}. {c}" for j, c in enumerate(item["choices"])
        )
        gold = labels[item["answer_idx"]]
    else:
        prompt = item["question"] + (BARE if item["qtype"] == "short" else "")
        gold = item["answer"]
    return {
        "id": item["id"], "qtype": item["qtype"],
        "knowledge": item["knowledge"],
        "messages": [{"role":"user","content":prompt},
                     {"role":"assistant","content":gold}],
        "answer": gold,
    }

def stratified_split(raw, rng, counts):
    by_type = {}
    for it in raw:
        by_type.setdefault(it["qtype"], []).append(it)
    total = N_TRAIN + N_VAL + N_TEST
    train, val, test = [], [], []
    for qt, pool in by_type.items():
        rng.shuffle(pool)
        n_total = counts[qt]
        n_test  = max(1, round(N_TEST * n_total / total))
        n_val   = max(1, round(N_VAL  * n_total / total))
        n_train = n_total - n_test - n_val
        need = n_test + n_val + n_train
        if len(pool) < need:
            raise ValueError(f"Not enough {qt} items: have {len(pool)}, need {need}")
        test += pool[:n_test]
        val  += pool[n_test:n_test+n_val]
        train+= pool[n_test+n_val:n_test+n_val+n_train]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

def freeze(examples, path):
    data = json.dumps(examples, indent=2, sort_keys=True)
    Path(path).write_text(data)
    return hashlib.sha256(data.encode()).hexdigest()

def main():
    rng = random.Random(SEED)
    Path("data").mkdir(exist_ok=True)

    players = load_batting()
    if not players:
        raise SystemExit("No Lahman players loaded — set BATTING_CSV or install pybaseball.")

    handwritten = json.loads(Path(HANDWRITTEN_PATH).read_text())

    TOTAL = N_TRAIN + N_VAL + N_TEST
    counts = {
        "mc":    round(TOTAL * MIX["mc"]),
        "short": round(TOTAL * MIX["short"]),
    }
    # (stats slice intentionally omitted for the pilot)

    raw = handwritten
    for j, it in enumerate(raw):
        it.setdefault("id", f"{it['qtype']}_{j:03d}")

    train, val, test = stratified_split(raw, rng, counts)

    train_ex = [to_example(it) for it in train]
    val_ex   = [to_example(it) for it in val]
    test_ex  = [to_example(it) for it in test]

    h_train = freeze(train_ex, "data/train.json")
    h_val   = freeze(val_ex,   "data/val.json")
    h_test  = freeze(test_ex,  "data/test.json")

    manifest = {
        "seed": SEED, "mix": MIX,
        "counts": {"train": len(train_ex), "val": len(val_ex), "test": len(test_ex)},
        "by_type": {
            "train": {qt: sum(1 for e in train_ex if e["qtype"]==qt) for qt in MIX},
            "test":  {qt: sum(1 for e in test_ex  if e["qtype"]==qt) for qt in MIX},
        },
        "hashes": {"train": h_train, "val": h_val, "test": h_test},
        "lahman_source": BATTING_CSV or "pybaseball",
    }
    Path("data/manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))

if __name__ == "__main__":
    main()

{
  "seed": 0,
  "mix": {
    "mc": 0.65,
    "short": 0.35
  },
  "counts": {
    "train": 24,
    "val": 3,
    "test": 9
  },
  "by_type": {
    "train": {
      "mc": 15,
      "short": 9
    },
    "test": {
      "mc": 6,
      "short": 3
    }
  },
  "hashes": {
    "train": "70d6d6854223e846cfbcbc65c2fbf649f570c200118b30b64eb4e3cdf3855ac2",
    "val": "2aac9699ee16feaf2f28c1f442991389a5c498d680e7c176c446f561e841d9c9",
    "test": "7871dc75ce595b350bc46b95acab47cbaead584fb8cceb11dfe2c2b5f7b23f9f"
  },
  "lahman_source": "/content/sample_data/Batting.csv"
}


In [13]:
"""
overlap_check.py — exact + near-duplicate overlap between train and test.
Works on the frozen JSON dicts written by build_dataset.py:
each item is {"id", "qtype", "knowledge", "messages":[{role,content},...], "answer"}.
"""
import json, sys
from difflib import SequenceMatcher
from pathlib import Path
from scorer import normalize


def _question(item):
    return item["messages"][0]["content"]


def exact_overlap(train, test):
    tr = {normalize(_question(e)) for e in train}
    return [t["id"] for t in test if normalize(_question(t)) in tr]


def near_duplicates(train, test, threshold=0.90):
    tr_q = [(e["id"], normalize(_question(e))) for e in train]
    hits = []
    for te in test:
        q = normalize(_question(te))
        for tid, tq in tr_q:
            if SequenceMatcher(None, q, tq).ratio() >= threshold:
                hits.append((te["id"], tid))
                break
    return hits


def report(train, test):
    ex = exact_overlap(train, test)
    nd = near_duplicates(train, test)
    print(f"Exact overlaps: {len(ex)}  {ex[:5]}")
    print(f"Near duplicates (>=0.90): {len(nd)}  {nd[:5]}")
    return {"exact": ex, "near": nd}


if __name__ == "__main__":
    train = json.loads(Path("data/train.json").read_text())
    test  = json.loads(Path("data/test.json").read_text())
    res = report(train, test)

    mpath = Path("data/manifest.json")
    m = json.loads(mpath.read_text())
    m["overlap"] = {"exact": len(res["exact"]), "near": len(res["near"]),
                    "exact_ids": res["exact"][:10], "near_pairs": res["near"][:10]}
    mpath.write_text(json.dumps(m, indent=2))
    print("Logged overlap counts into data/manifest.json")


Exact overlaps: 0  []
Near duplicates (>=0.90): 1  [('short_038', 'short_037')]
Logged overlap counts into data/manifest.json


In [17]:
# eval_test.py — evaluate a model on data/test.json, score per-type
import os, json, time, torch
from pathlib import Path
from dataclasses import asdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata
from scorer import score

MODEL_ID = "google/gemma-3-1b-it"
HF_TOKEN = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or "PASTE_YOUR_TOKEN_HERE"
TEST_PATH = "data/test.json"
RUNS_PATH = "runs.jsonl"

from dataclasses import dataclass, field
import datetime

@dataclass
class RunLog:
    run_id: str
    method: str
    rank: int = 0
    target_modules: list = field(default_factory=list)
    quantization: str | None = None
    precision: str = "fp16"
    seed: int = 0
    test_accuracy_overall: float = 0.0
    test_accuracy_mc: float = 0.0
    test_accuracy_short: float = 0.0
    test_accuracy_stats: float = 0.0
    gen_latency_sec_64tok: float = 0.0
    peak_gpu_mem_mb: float = 0.0
    model_id: str = MODEL_ID
    timestamp_utc: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat()+"Z")
    notes: str = ""

    def save(self, path=RUNS_PATH):
        with open(path, "a") as f:
            f.write(json.dumps(asdict(self)) + "\n")

test = json.loads(Path(TEST_PATH).read_text())
print(f"Loaded {len(test)} test items:",
      {qt: sum(1 for e in test if e['qtype']==qt) for qt in {e['qtype'] for e in test}})

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map="auto")
model.eval()

def generate(messages, max_new_tokens=96):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs["input_ids"], max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

by_type, results = {}, []
t0 = time.time()
for e in test:
    pred = generate(e["messages"][:1])
    ok = score(pred, e["answer"], e["qtype"])
    by_type.setdefault(e["qtype"], []).append(ok)
    results.append({"id": e["id"], "qtype": e["qtype"],
                    "gold": e["answer"], "pred": pred, "correct": ok})
elapsed = time.time() - t0

def acc(lst): return sum(lst)/len(lst) if lst else 0.0
overall = sum(r["correct"] for r in results)/len(results)
accs = {qt: acc(v) for qt, v in by_type.items()}

print(f"\nOverall: {overall:.0%}  ({sum(r['correct'] for r in results)}/{len(results)})  [{elapsed:.1f}s]")
for qt, a in sorted(accs.items()):
    print(f"  {qt:6}: {a:.0%}")
for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"  {mark} [{r['qtype']:5}] gold={r['gold']!r:>14}  pred={r['pred'][:60]!r}")

prompts = [e["messages"][0]["content"] for e in test[:5]]
t0 = time.time()
for p in prompts:
    generate([{"role":"user","content":p}], max_new_tokens=64)
lat = (time.time()-t0)/len(prompts)
peak = torch.cuda.max_memory_allocated()/1e6

log = RunLog(run_id="base_zero_shot", method="base",
             test_accuracy_overall=overall,
             test_accuracy_mc=accs.get("mc",0.0),
             test_accuracy_short=accs.get("short",0.0),
             test_accuracy_stats=accs.get("stats",0.0),
             gen_latency_sec_64tok=lat, peak_gpu_mem_mb=peak,
             notes=f"n={len(results)}; pilot test set")
log.save()
print(f"\nLatency ~{lat:.2f}s/prompt (64 tok). Peak GPU {peak/1e3:.2f} GB.")
print(f"RunLog written to {RUNS_PATH}")

# save raw generations for inspection
Path("data/base_generations.json").write_text(json.dumps(results, indent=2))

Loaded 9 test items: {'short': 3, 'mc': 6}


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


Overall: 44%  (4/9)  [28.6s]
  mc    : 33%
  short : 67%
  ✓ [short] gold='Wrigley Field'  pred='Wrigley Field'
  ✗ [short] gold=   'Pete Rose'  pred='Babe Ruth'
  ✗ [mc   ] gold=           'B'  pred="The correct answer is **D. Live ball**.\n\nHere's why:\n\n*   **"
  ✓ [mc   ] gold=           'B'  pred="The correct answer is **B. 4**\n\nHere's why:\n\n*   A runner mu"
  ✓ [mc   ] gold=           'C'  pred="The correct answer is **C. 162**\n\nHere's why:\n\n*   **MLB Reg"
  ✗ [mc   ] gold=           'A'  pred='The correct answer is **C. Houston Astros**.\n\nThe Houston As'
  ✓ [short] gold='Perfect game'  pred='Perfect Game'
  ✗ [mc   ] gold=           'B'  pred="The correct answer is **A. Triple**.\n\nHere's why:\n\n*   **Tri"
  ✗ [mc   ] gold=           'C'  pred="The correct answer is **B. 2**\n\nHere's why:\n\n*   A half-inni"

Latency ~2.31s/prompt (64 tok). Peak GPU 4.02 GB.
RunLog written to runs.jsonl


/tmp/ipykernel_525/3693860546.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp_utc: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat()+"Z")


2542